In [1]:
"""
HaGRID 데이터셋으로 YOLOv10n 1차 학습 스크립트

사용 흐름:
    1. config 설정 (클래스, 이미지 개수)
    2. python step1_download_hagrid.py   → HaGRID 원하는 클래스만 다운로드
    3. python step2_convert_to_yolo.py   → YOLO 포맷으로 변환 + 이미지 개수 제한
    4. python step3_train.py             → YOLOv10n 1차 학습

이 파일은 통합 버전 (한 번에 실행 가능)
"""

'\nHaGRID 데이터셋으로 YOLOv10n 1차 학습 스크립트\n \n사용 흐름:\n    1. config 설정 (클래스, 이미지 개수)\n    2. python step1_download_hagrid.py   → HaGRID 원하는 클래스만 다운로드\n    3. python step2_convert_to_yolo.py   → YOLO 포맷으로 변환 + 이미지 개수 제한\n    4. python step3_train.py             → YOLOv10n 1차 학습\n \n이 파일은 통합 버전 (한 번에 실행 가능)\n'

In [2]:
import os
import json
import random
import shutil
import subprocess
from pathlib import Path

## config

In [3]:
# 1. 사용할 클래스 목록 (HaGRID 공식 클래스명 기준)
TARGET_CLASSES = [
    "palm",
    "fist",
    "ok",
    "three",
    "like",
    "call",
    "rock",
    "three2",
    "three_gun"
]
# 2. 클래스당 최대 이미지 개수 (None이면 전체 다 사용)
MAX_IMAGES_PER_CLASS = 500

# 3. train/val/test 비율
SPLIT_RATIO = {"train": 0.7, "val": 0.2, "test": 0.1}

# 4. 경로 설정
HAGRID_REPO_DIR = "./hagrid_repo"          # HaGRID 공식 repo 클론 위치
RAW_DATA_DIR = "./data/hagrid_raw"          # 다운로드한 원본 이미지+어노테이션
YOLO_DATA_DIR = "./data/hagrid_yolo"        # YOLO 포맷 변환 결과
SEED = 42

## 1. HaGRID repo 클론

In [4]:
def clone_hagrid_repo():
    if os.path.exists(HAGRID_REPO_DIR):
        print(f"[skip] 이미 존재함: {HAGRID_REPO_DIR}")
        return
    print("[1/5] HaGRID 공식 repo 클론 중...")
    subprocess.run(
        ["git", "clone", "https://github.com/hukenovs/hagrid.git", HAGRID_REPO_DIR],
        check=True,
    )
    subprocess.run(
    ["pip", "install", "requests", "tqdm"],
    check=True,
    )

## 2. 필요한 클래스 다운로드

In [5]:
def download_hagrid_classes():
    print(f"[2/5] HaGRID 클래스 다운로드: {TARGET_CLASSES}")
    os.makedirs(RAW_DATA_DIR, exist_ok=True)

    download_script = os.path.join(HAGRID_REPO_DIR, "download.py")

    cmd = [
        "python", download_script,
        "--save_path", RAW_DATA_DIR,
        "--annotations",
        "--dataset",
        "--targets", *TARGET_CLASSES,
    ]
    print(" ".join(cmd))
    subprocess.run(cmd, check=True)

## 3. 포맷 변환

In [6]:
def normalize_bbox_to_yolo(bbox):
    """
    HaGRID bbox: [top_left_x, top_left_y, width, height] (0~1 정규화됨)
    YOLO format: [center_x, center_y, width, height] (0~1 정규화됨)
    """
    x, y, w, h = bbox
    cx = x + w / 2
    cy = y + h / 2
    return cx, cy, w, h


def convert_and_limit():
    print("[3/5] YOLO 포맷 변환 + 이미지 개수 제한")
    random.seed(SEED)

    class_to_id = {cls: idx for idx, cls in enumerate(TARGET_CLASSES)}

    # 출력 디렉토리 준비
    for split in SPLIT_RATIO:
        os.makedirs(f"{YOLO_DATA_DIR}/images/{split}", exist_ok=True)
        os.makedirs(f"{YOLO_DATA_DIR}/labels/{split}", exist_ok=True)

    ann_dir = os.path.join(RAW_DATA_DIR, "hagrid_annotations", "train")  # HaGRID는 통합 어노테이션 디렉토리 구조 사용
    img_root = os.path.join(RAW_DATA_DIR, "hagrid_dataset")

    for cls in TARGET_CLASSES:
        ann_path = os.path.join(ann_dir, f"{cls}.json")
        if not os.path.exists(ann_path):
            print(f"  [경고] 어노테이션 없음, 스킵: {ann_path}")
            continue

        with open(ann_path, "r") as f:
            annotations = json.load(f)

        image_ids = list(annotations.keys())
        random.shuffle(image_ids)

        if MAX_IMAGES_PER_CLASS is not None:
            image_ids = image_ids[:MAX_IMAGES_PER_CLASS]

        print(f"  - {cls}: {len(image_ids)}장 사용")

        # split 분배
        n = len(image_ids)
        n_train = int(n * SPLIT_RATIO["train"])
        n_val = int(n * SPLIT_RATIO["val"])
        split_map = (
            [("train", i) for i in image_ids[:n_train]]
            + [("val", i) for i in image_ids[n_train:n_train + n_val]]
            + [("test", i) for i in image_ids[n_train + n_val:]]
        )

        for split, img_id in split_map:
            ann = annotations[img_id]
            src_img = os.path.join(img_root, cls, f"{img_id}.jpg")
            if not os.path.exists(src_img):
                continue

            dst_img = f"{YOLO_DATA_DIR}/images/{split}/{cls}_{img_id}.jpg"
            dst_label = f"{YOLO_DATA_DIR}/labels/{split}/{cls}_{img_id}.txt"

            shutil.copy(src_img, dst_img)

            lines = []
            for bbox, label in zip(ann["bboxes"], ann["labels"]):
                if label not in class_to_id:
                    continue  # no_gesture 등 타겟 외 클래스 제외
                cx, cy, w, h = normalize_bbox_to_yolo(bbox)
                class_id = class_to_id[label]
                lines.append(f"{class_id} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")

            with open(dst_label, "w") as f:
                f.write("\n".join(lines))

## 4. dataset.yaml 생성

In [7]:
def create_yaml():
    print("[4/5] dataset.yaml 생성")
    yaml_content = f"""path: {os.path.abspath(YOLO_DATA_DIR)}
train: images/train
val: images/val
test: images/test

names:
"""
    for idx, cls in enumerate(TARGET_CLASSES):
        yaml_content += f"  {idx}: {cls}\n"

    yaml_path = f"{YOLO_DATA_DIR}/dataset.yaml"
    with open(yaml_path, "w") as f:
        f.write(yaml_content)
    print(f"  생성됨: {yaml_path}")
    return yaml_path

In [ ]:
if __name__ == "__main__":
    clone_hagrid_repo()
    download_hagrid_classes()
    convert_and_limit()
    yaml_path = create_yaml()
    print(f"\n완료. dataset.yaml 위치: {yaml_path}")

[1/5] HaGRID 공식 repo 클론 중...
[2/5] HaGRID 클래스 다운로드: ['palm', 'fist', 'ok', 'three', 'like', 'call', 'rock', 'three2', 'three_gun']
python ./hagrid_repo/download.py --save_path ./data/hagrid_raw --annotations --dataset --targets palm fist ok three like call rock three2 three_gun
